# Project 4 手把拍照與道路目標標記

保留 Project 3 車道辨識，按手把 13 號鍵拍一張「未畫線原圖」，以該幀綠點為目標標籤；長按不連拍。

預設按 5 號鍵以速度 0.3 前進 1 秒，1 號鍵停止。按鍵名稱依手把而異，請看 Controller 的編號；這是手動定時移動，沒有自動循線或新增轉向。Run All 不會自動開車。

新資料在 `project4_data/dataset_xy`，辨識檢查圖在 `project4_data/review`。不要把紅線／綠點畫進訓練圖，也不要混用 Project 3 的舊 dataset_xy。

先開始辨識，喚醒手把，再按「綁定手把」。確認綠點正確才按拍照；辨識失敗時暫停，點左側原圖的道路目標即可手動標記並儲存。累積目標 200 張，達標仍可繼續拍。


## 1. 載入套件
保留原檔提供的初始化程式。Notebook 使用 Python 3.6 可解析的語法。


In [ ]:
"""Provided setup cell. Do not modify."""
# Notebook display and widget utilities.
import ipywidgets
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

# JetBot camera and robot interfaces.
from jetbot import Robot, Camera, bgr8_to_jpeg

# General utilities and image-processing packages.
from uuid import uuid1
import os
import json
import glob
import datetime
import numpy as np
import cv2
import time

import hashlib


## 2. 初始化相機與資料集
保留原檔提供的設定，既有照片不刪除。相機初始化每個 Kernel 執行一次；停止相機後若要重啟，請重新啟動 Kernel。


In [ ]:
"""Provided setup cell. Do not modify."""
from jupyter_clickable_image_widget import ClickableImageWidget

DATASET_DIR = os.path.join("project4_data", "dataset_xy")

# Create the dataset directory; keep existing files if it already exists.
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print("Directories not created because they already exist")

camera = Camera()


## 3. 室內膠帶参数

`LANE_PROFILE` 預設為 `INDOOR_PROFILE`。工作寬度改為 480 px，ROI 從高度 12% 開始，涵蓋下方兩側膠帶；不要求膠帶為低飽和白色，也不要求朝向固定消失點。

- `contrast=8`：膠帶相對鄰近地墊的亮度差。
- `min_gray=60`：允許偏暗／偏紫的膠帶。
- `component_min_height=0.10`：排除地墊紋理等零碎亮點。
- `fit_tolerance=8`：曲線擬合容許離群列的像素差。
- `max_draw_gap=0.025`：長斷口不補線。

`MAX_FPS=5` 僅為排程上限。保留 `OUTDOOR_PROFILE_3` 與舊函式供比較，若要恢復原設定可改為 `LANE_PROFILE = dict(OUTDOOR_PROFILE_3)`。

Hough：門檻 12、最短線段 12 px、允許斷距 10 px（480 px 工作圖）。`CENTERLINE_CONFIG` 可調整固定近端位置與遠端觀察高度；預設近端為 (50%, 95%)，遠端高度 35%。


In [ ]:
OUTDOOR_PROFILE_3 = {'roi': [[0, 0.985], [0.465, 0.565], [0.51, 0.565], [1, 0.965], [1, 1], [0, 1]],
 'vanishing_point': [0.49, 0.55],
 'vp_tolerance': 0.025,
 'contrast': 12,
 'min_gray': 100,
 'canny': [35, 100]}

# 室內地墊膠帶場景：允許色偏及彎曲標線，不沿用戶外照片的消失點。
INDOOR_PROFILE = {
    'name': 'indoor_tape',
    'working_width': 480,
    'roi': [[0.0, 0.98], [0.0, 0.45], [0.28, 0.12],
            [0.72, 0.12], [1.0, 0.45], [1.0, 0.98]],
    'contrast': 8,
    'min_gray': 60,
    'background_sigma_x': 15,
    'background_sigma_y': 3,
    'component_min_height': 0.10,
    'component_min_area': 0.0004,
    'min_row_width': 3,
    'max_row_width': 42,
    'fit_tolerance': 8,
    'min_lane_height': 0.25,
    'max_draw_gap': 0.025,
    'hough_threshold': 12,
    'hough_min_length': 12,
    'hough_max_gap': 10,
}



LANE_PROFILE = dict(INDOOR_PROFILE)
MAX_FPS = 5.0
SHOW_MIDPOINT = True

# 座標為畫面寬／高比例：y 越大越靠近車子。
CENTERLINE_CONFIG = {
    'near_x': 0.5,
    'near_y': 0.95,
    'far_y': 0.65,
    'min_lane_width': 0.10,
}

# 課堂範例使用 13 號鍵拍照；不同手把請依 Controller 顯示調整。
GAMEPAD_INDEX = 0
CAPTURE_BUTTON = 13
MOVE_BUTTON = 5
STOP_BUTTON = 1
MOVE_SPEED = 0.3
MOVE_SECONDS = 1.0
TARGET_COUNT = 200
MAX_CAPTURE_AGE = 1.5
CAPTURE_DEBOUNCE = 0.4


## 4 車道辨識函式
保留原本室內膠帶參數、Canny、HoughLinesP 與曲線整理。藍點固定在高度 95%，綠點為高度 65% 的左右中點。這是標籤建議，需由拍攝者判斷是否為合理行進目標。全部沿用固定高度時，y 標籤幾乎不變，模型主要學習 x；若彎道需要不同前視距離，可暫停後手動點選，並維持一致標記規則。


In [ ]:
def detect_profile3(image: np.ndarray, profile: dict, *, use_morphology: bool = True):
    """回傳原解析度疊圖、中間影像、實際偵測座標與統計。"""
    original_h, original_w = image.shape[:2]
    w = 1000
    h = round(original_h * w / original_w)
    frame = cv2.resize(image, (w, h), interpolation=cv2.INTER_AREA if original_w > w else cv2.INTER_LINEAR)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    smooth = cv2.GaussianBlur(gray, (3, 3), 0)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # 局部亮度差保留比周圍路面明亮的窄標線，避免固定灰階 160 漏掉陰影下標線。
    local = cv2.GaussianBlur(smooth, (31, 31), 0)
    contrast = cv2.subtract(smooth, local)
    white = (smooth >= profile['min_gray']) & (hsv[:, :, 1] <= 110)
    yellow = (hsv[:, :, 0] >= 10) & (hsv[:, :, 0] <= 40) & (hsv[:, :, 1] >= 35) & (hsv[:, :, 2] >= 95)
    binary = np.uint8((contrast >= profile['contrast']) & (white | yellow)) * 255
    roi = np.zeros_like(gray)
    polygon = np.rint(np.array(profile['roi']) * [w - 1, h - 1]).astype(np.int32)
    cv2.fillPoly(roi, [polygon], 255)
    # 白色車身比標線寬：以面積及厚度辨認亮色塊，排除其外接框。
    # 這是針對本作業的遮擋抑制，並非通用車輛辨識器。
    broad_white = np.uint8((smooth > 175) & (hsv[:, :, 1] < 70)) * 255
    broad_white = cv2.morphologyEx(broad_white, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    _, _, stats, _ = cv2.connectedComponentsWithStats(broad_white)
    occlusion = np.zeros_like(gray)
    for bx, by, bw, bh, area in stats[1:]:
        if 12 <= bw <= .12*w and 8 <= bh <= .12*h and area >= 80 and profile['vanishing_point'][1]*h - 8 <= by <= .62*h:
            cv2.rectangle(occlusion, (max(0, bx-9), max(0, by-6)), (min(w-1, bx+bw+12), min(h-1, by+bh+10)), 255, -1)
    roi = cv2.bitwise_and(roi, cv2.bitwise_not(occlusion))
    binary = cv2.bitwise_and(binary, roi)
    # 只填補小缺口；不在單像素邊緣上開運算，以免遠處細線消失。
    morphology = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8)) if use_morphology else binary.copy()
    edges = cv2.Canny(smooth, *profile['canny'])
    candidates = cv2.bitwise_and(edges, cv2.dilate(morphology, np.ones((3, 3), np.uint8)))
    candidates = cv2.bitwise_and(candidates, roi)
    paint_support = cv2.dilate(morphology, np.ones((5, 5), np.uint8))
    raw = cv2.HoughLinesP(candidates, 1, np.pi / 180, 12, minLineLength=12, maxLineGap=10)
    raw = [] if raw is None else raw.reshape(-1, 4).tolist()
    vx, vy = np.array(profile['vanishing_point']) * [w, h]
    segments = []
    candidate_view = frame.copy()
    for x1, y1, x2, y2 in raw:
        cv2.line(candidate_view, (x1, y1), (x2, y2), (0, 165, 255), 1, cv2.LINE_AA)
        dx, dy = x2 - x1, y2 - y1
        if abs(dy) < 4 or abs(dy) / max(abs(dx), 1) < .15:
            continue
        a = dx / dy
        b = x1 - a * y1
        if abs(a * vy + b - vx) > w * profile['vp_tolerance']:
            continue
        # 確認沿線附近確實有標線像素支持，不只是一條路面刮痕邊緣。
        n = max(2, int(np.hypot(dx, dy)))
        xs = np.clip(np.rint(np.linspace(x1, x2, n)).astype(int), 0, w - 1)
        ys = np.clip(np.rint(np.linspace(y1, y2, n)).astype(int), 0, h - 1)
        if np.mean(paint_support[ys, xs] > 0) < .65:
            continue
        segments.append({'points': [x1, y1, x2, y2], 'bottom': a * (h - 1) + b, 'slope': a, 'length': float(np.hypot(dx, dy))})

    # 同一條標線的兩側邊緣及重複 Hough 線段合併，保留多車道及垂直中央線。
    groups = []
    def same_marking(s, anchor):
        if abs(s['bottom'] - anchor['bottom']) < 55 and abs(s['slope'] - anchor['slope']) < .50:
            return True
        ax, ay = anchor['points'][:2]
        endpoints = np.array(s['points']).reshape(2, 2)
        distance = np.abs(endpoints[:, 0] - ax - anchor['slope'] * (endpoints[:, 1] - ay)) / np.sqrt(1 + anchor['slope'] ** 2)
        return np.max(distance) < 8 and abs(np.arctan(s['slope']) - np.arctan(anchor['slope'])) < .10

    for s in sorted(segments, key=lambda s: -s['length']):
        match = next((g for g in groups if same_marking(s, g[0])), None)
        if match is None:
            groups.append([s])
        else:
            match.append(s)
    overlay = image.copy()
    detected_mask = np.zeros((original_h, original_w), np.uint8)
    lanes = []
    for group in groups:
        points = np.array([s['points'] for s in group]).reshape(-1, 2)
        if np.ptp(points[:, 1]) < 12 or sum(s['length'] for s in group) < 60:
            continue
        coeff = np.polyfit(points[:, 1], points[:, 0], 1)
        ys_all, xs_all = np.nonzero(morphology)
        distance = np.abs(xs_all - np.polyval(coeff, ys_all)) / np.sqrt(1 + coeff[0] ** 2)
        near = (distance < 8) & (ys_all >= vy + 5) & (ys_all <= points[:, 1].max() + 5)
        xs, ys = xs_all[near], ys_all[near]
        if len(xs) < 20:
            continue
        row_y = np.unique(ys)
        row_x = np.array([np.median(xs[ys == y]) for y in row_y])
        degree = 2 if np.ptp(row_y) > 45 and len(row_y) > 30 else 1
        coeff = np.polyfit(row_y, row_x, degree)
        # 按像素支持區間繪製，虛線缺口與車輛遮擋不補成整條實線。
        row_parts = np.split(row_y, np.flatnonzero(np.diff(row_y) > 3) + 1)
        paths = []
        for part in row_parts:
            if len(part) < 3 or part[-1] - part[0] < 2:
                continue
            y = np.arange(part[0], part[-1] + 1, dtype=float)
            x = np.polyval(coeff, y)
            px = np.clip(np.rint(x).astype(int), 0, w-1)
            py = y.astype(int)
            valid = (x >= 0) & (x < w) & (roi[py, px] > 0) & (paint_support[py, px] > 0)
            ids = np.flatnonzero(valid)
            for run in np.split(ids, np.flatnonzero(np.diff(ids) > 1) + 1):
                if len(run) < 3:
                    continue
                xy = np.column_stack([x[run] * original_w / w, y[run] * original_h / h])
                xy = np.rint(xy).astype(np.int32)
                xy[:, 0] = np.clip(xy[:, 0], 0, original_w - 1)
                xy[:, 1] = np.clip(xy[:, 1], 0, original_h - 1)
                thickness = max(2, round(original_w / 400))
                cv2.polylines(overlay, [xy], False, (0, 0, 255), thickness, cv2.LINE_AA)
                cv2.polylines(detected_mask, [xy], False, 255, thickness)
                paths.append(xy.tolist())
        if paths:
            lanes.append({'paths': paths, 'hough_support': len(group)})
    stages = {'gray': gray, 'filter': smooth, 'contrast': contrast, 'bin': binary, 'roi': roi, 'occlusion': occlusion,
              'morphology': morphology, 'edge': edges, 'candidate_edge': candidates, 'hough': candidate_view,
              'detected_mask': detected_mask}
    metadata = {'input_size': [original_w, original_h], 'working_size': [w, h], 'profile': profile,
                'use_morphology': use_morphology, 'raw_hough_count': len(raw), 'selected_hough_count': len(segments),
                'lane_group_count': len(lanes), 'drawn_pixel_count': int(np.count_nonzero(detected_mask)), 'lanes': lanes}
    return overlay, stages, metadata


def detect_indoor(image, profile, use_morphology=True):
    """以局部亮帶、連通區域與穩健二次擬合偵測左右膠帶；回傳格式與 detect 相同。"""
    if not isinstance(image, np.ndarray) or image.ndim != 3 or image.shape[2] != 3:
        raise ValueError('輸入必須是 BGR 彩色影像')
    original_h, original_w = image.shape[:2]
    w = int(profile['working_width'])
    h = max(2, int(round(original_h * w / float(original_w))))
    frame = cv2.resize(image, (w, h), interpolation=cv2.INTER_AREA if original_w > w else cv2.INTER_LINEAR)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    smooth = cv2.GaussianBlur(gray, (5, 5), 0)
    background = cv2.GaussianBlur(smooth, (0, 0),
                                  profile['background_sigma_x'], sigmaY=profile['background_sigma_y'])
    contrast = cv2.subtract(smooth, background)
    # 不使用 HSV 白色低飽和限制：畫面色偏時膠帶仍比鄰近地墊亮。
    binary = np.uint8((contrast >= profile['contrast']) &
                      (smooth >= profile['min_gray'])) * 255
    roi = np.zeros_like(gray)
    polygon = np.rint(np.asarray(profile['roi']) * [w - 1, h - 1]).astype(np.int32)
    cv2.fillPoly(roi, [polygon], 255)
    binary = cv2.bitwise_and(binary, roi)
    morphology = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8)) if use_morphology else binary.copy()
    # 地墊網格的亮點不應成為標線，先排除矮小／零碎區域。
    _, labels, stats, _ = cv2.connectedComponentsWithStats(morphology)
    clean = np.zeros_like(gray)
    components = 0
    for label in range(1, len(stats)):
        x, y, bw, bh, area = stats[label]
        if bh >= profile['component_min_height'] * h and area >= profile['component_min_area'] * w * h:
            clean[labels == label] = 255
            components += 1

    # PPT 必要流程：二值候選邊缘 -> HoughLinesP -> 依線段支持篩選。
    # linesP 直接保存 Hough 的原始輸出；它實際決定哪些候選列可進入曲線擬合。
    edge = cv2.Canny(clean, 50, 150)
    linesP = cv2.HoughLinesP(edge, 1, np.pi / 180, profile['hough_threshold'],
                             minLineLength=profile['hough_min_length'],
                             maxLineGap=profile['hough_max_gap'])
    raw_lines = [] if linesP is None else linesP.reshape(-1, 4).tolist()
    hough_support = np.zeros_like(gray)
    selected_lines = []
    hough_preview = frame.copy()
    for x1, y1, x2, y2 in raw_lines:
        dx, dy = x2 - x1, y2 - y1
        if abs(dy) < 4 or abs(dy) / max(abs(dx), 1) < 0.15:
            continue
        selected_lines.append([x1, y1, x2, y2])
        cv2.line(hough_support, (x1, y1), (x2, y2), 255, 9)
        cv2.line(hough_preview, (x1, y1), (x2, y2), (0, 165, 255), 2, cv2.LINE_AA)

    overlay = image.copy()
    lane_mask = np.zeros((original_h, original_w), np.uint8)
    lanes = []
    selected_rows = 0
    for side in ('left', 'right'):
        samples = []
        for y in range(h):
            xs = np.flatnonzero(clean[y])
            xs = xs[xs < w / 2.0] if side == 'left' else xs[xs >= w / 2.0]
            if not len(xs):
                continue
            runs = np.split(xs, np.flatnonzero(np.diff(xs) > 1) + 1)
            runs = [run for run in runs if profile['min_row_width'] <= len(run) <= profile['max_row_width']]
            runs = [run for run in runs if np.any(hough_support[y, run] > 0)]
            if not runs:
                continue
            # 選道路內側的連續亮帶，避免選到更外側的牆面反光。
            run = max(runs, key=lambda r: r.mean()) if side == 'left' else min(runs, key=lambda r: r.mean())
            samples.append((float(np.median(run)), y))
        if len(samples) < max(20, int(.15 * h)):
            continue
        points = np.asarray(samples)
        x, y = points[:, 0], points[:, 1]
        if np.ptp(y) < profile['min_lane_height'] * h:
            continue
        # x=f(y) 可保留近垂直邊線；反覆剔除離群列，避免牆角／地墊接縫拉歪曲線。
        keep = np.ones(len(y), dtype=bool)
        for _ in range(4):
            if np.count_nonzero(keep) < 20:
                break
            coef = np.polyfit(y[keep] / h, x[keep], 2)
            residual = np.abs(x - np.polyval(coef, y / h))
            keep = residual <= profile['fit_tolerance']
        if np.count_nonzero(keep) < max(20, int(.15 * h)) or np.ptp(y[keep]) < profile['min_lane_height'] * h:
            continue
        coef = np.polyfit(y[keep] / h, x[keep], 2)
        support_y = y[keep].astype(int)
        paths = []
        max_gap = max(2, int(round(profile['max_draw_gap'] * h)))
        for rows in np.split(support_y, np.flatnonzero(np.diff(support_y) > max_gap) + 1):
            if len(rows) < 8:
                continue
            yy = np.arange(rows[0], rows[-1] + 1)
            xx = np.polyval(coef, yy / float(h))
            px = np.clip(np.rint(xx).astype(int), 0, w - 1)
            # 只在 ROI 內画線；不越過長遮擋區間補出一整條假線。
            valid = (xx >= 0) & (xx < w) & (roi[yy, px] > 0)
            indices = np.flatnonzero(valid)
            for chunk in np.split(indices, np.flatnonzero(np.diff(indices) > 1) + 1):
                if len(chunk) < 8:
                    continue
                xy = np.rint(np.column_stack((xx[chunk] * original_w / w,
                                             yy[chunk] * original_h / h))).astype(np.int32)
                xy[:, 0] = np.clip(xy[:, 0], 0, original_w - 1)
                xy[:, 1] = np.clip(xy[:, 1], 0, original_h - 1)
                thickness = max(2, int(round(original_w / 220.0)))
                cv2.polylines(overlay, [xy], False, (0, 0, 255), thickness, cv2.LINE_AA)
                cv2.polylines(lane_mask, [xy], False, 255, thickness)
                paths.append(xy.tolist())
        if paths:
            selected_rows += int(np.count_nonzero(keep))
            lanes.append({'side': side, 'paths': paths, 'supported_rows': int(np.count_nonzero(keep))})
    stages = {'gray': gray, 'filter': smooth, 'contrast': contrast, 'bin': binary,
              'roi': roi, 'morphology': morphology, 'candidate_mask': clean,
              'detected_mask': lane_mask, 'edge': edge,
              'hough_support': hough_support, 'hough': hough_preview}
    metadata = {'input_size': [original_w, original_h], 'working_size': [w, h],
                'profile': profile, 'lane_group_count': len(lanes), 'lanes': lanes,
                'candidate_component_count': components, 'selected_row_count': selected_rows,
                'drawn_pixel_count': int(np.count_nonzero(lane_mask)),
                'raw_hough_count': len(raw_lines), 'selected_hough_count': len(selected_lines),
                'raw_hough_lines': raw_lines}
    return overlay, stages, metadata


def detect(image, profile, use_morphology=True):
    if profile.get('name') == 'indoor_tape':
        return detect_indoor(image, profile, use_morphology=use_morphology)
    return detect_profile3(image, profile, use_morphology=use_morphology)


## 5 介面與按鍵
手把初始化與按鍵綁定分開；先顯示 Controller 並按任一鍵喚醒，再按綁定。拍照不要求勾選允許移動。前進需同時勾選允許移動且辨識正在執行。


In [ ]:
previous = globals().get('lane_controller')
if previous is not None:
    previous.close()

camera_widget = ClickableImageWidget(width=camera.width, height=camera.height)
img_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)
snapshot_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)
count_widget = ipywidgets.IntText(description='已存張數', disabled=True)
count_widget.value = len(glob.glob(os.path.join(DATASET_DIR, '*.jpg')))
status_widget = ipywidgets.Label(value='尚未開始，請按開始辨識。')
save_status_widget = ipywidgets.Label(value='可暫停後點左圖，或按儲存目前結果。')
start_button = ipywidgets.Button(description='開始辨識', button_style='success')
pause_button = ipywidgets.Button(description='暫停辨識', button_style='warning')
save_button = ipywidgets.Button(description='拍照存訓練原圖', button_style='info')
stop_camera_button = ipywidgets.Button(description='停止相機', button_style='danger')

controller = ipywidgets.Controller(index=GAMEPAD_INDEX)
bind_gamepad_button = ipywidgets.Button(description='綁定手把')
gamepad_status = ipywidgets.Label(value='請先按手把任一鍵，確認 Controller 回應後再綁定。')
move_enabled = ipywidgets.Checkbox(value=False, description='允許移動')
move_button = ipywidgets.Button(description='前進一次')
stop_move_button = ipywidgets.Button(description='停止移動', button_style='danger')
motion_status = ipywidgets.Label(value='未啟用移動')
progress_widget = ipywidgets.IntProgress(value=0, min=0, max=TARGET_COUNT, description='收集進度')
progress_text = ipywidgets.Label(value='0 / {} 張'.format(TARGET_COUNT))


## 6 同幀資料收集
按鍵儲存最近已發布影格的原始 JPEG，檔名為 `xy_<x>_<y>_<uuid>.jpg`。座標單位為原始圖片像素，可供課堂 XYDataset 讀取。

原圖、辨識結果、中點均源自同一幀。運行時拒絕超過 1.5 秒的影格，重複同幀不再存；暫停後可檢查畫面再拍。手動標記只允許暫停時點左圖，點一下即存一張。

這不是相機硬體快門，存的是按下時最近完成辨識的畫面，可能有運算／瀏覽器延遲；需要精準確認時停車再暫停辨識。確認右側檢查圖的黃色圈位置正確。


In [ ]:
import asyncio
import copy
from concurrent.futures import ThreadPoolExecutor


def draw_lane_midpoint(image, metadata):
    """近端固定；遠端取同一高度的左右線平均，不延用前幀或外推缺線。"""
    height, width = image.shape[:2]
    config = CENTERLINE_CONFIG
    near = (int(round((width - 1) * config['near_x'])),
            int(round((height - 1) * config['near_y'])))
    y_ref = int(round((height - 1) * config['far_y']))
    state = {'valid': False, 'near_point': list(near), 'far_point': None,
             'left_point': None, 'right_point': None, 'lookahead_y': y_ref,
             'reason': 'missing_lanes_at_lookahead'}
    metadata['centerline'] = state
    radius = max(3, width // 100)
    cv2.circle(image, near, radius, (255, 0, 0), -1)
    left, right = [], []
    for lane in metadata.get('lanes', []):
        samples = []
        for path in lane.get('paths', []):
            points = np.asarray(path, dtype=float)
            if points.ndim != 2 or len(points) < 2 or points.shape[1] != 2:
                continue
            if not np.isfinite(points).all():
                continue
            points = points[np.argsort(points[:, 1])]
            if points[0, 1] <= y_ref <= points[-1, 1]:
                samples.append(float(np.interp(y_ref, points[:, 1], points[:, 0])))
        if not samples:
            continue
        x = float(np.median(samples))
        if not 0 <= x < width:
            continue
        # 室內版已有左右標記，轉彎偏移時不以影像中心重新分類。
        side = lane.get('side')
        if side is None:
            side = 'left' if x < width / 2.0 else 'right'
        if side == 'left':
            left.append(x)
        elif side == 'right':
            right.append(x)
    if not left or not right:
        return None
    lx, rx = max(left), min(right)
    if rx - lx < config['min_lane_width'] * width:
        state['reason'] = 'crossed_or_too_narrow'
        return None
    midpoint = (int(round((lx + rx) / 2.0)), y_ref)
    state.update(valid=True, far_point=list(midpoint), left_point=[lx, y_ref],
                 right_point=[rx, y_ref], reason='both_lanes_detected')
    cv2.line(image, near, midpoint, (0, 0, 255), max(2, width // 250), cv2.LINE_AA)
    cv2.circle(image, midpoint, radius, (0, 255, 0), -1)
    cv2.circle(image, near, radius, (255, 0, 0), -1)
    return midpoint


def process_camera_frame(frame, profile, show_midpoint):
    """背景執行緒只計算影像；不在背景執行緒修改 widget。"""
    line_img, _, metadata = detect(frame, profile)
    midpoint = draw_lane_midpoint(line_img, metadata) if show_midpoint else None
    metadata['midpoint'] = list(midpoint) if midpoint is not None else None
    return bytes(bgr8_to_jpeg(frame)), bytes(bgr8_to_jpeg(line_img)), metadata


class LiveLaneController:
    """每次最多處理一幀，不排隊累積過期影格；畫面更新回到 Notebook 事件迴圈。"""

    def __init__(self, camera, raw_widget, result_widget, saved_widget,
                 count_widget, status_widget, save_status_widget, dataset_dir,
                 profile, max_fps=5.0, show_midpoint=True, processor=None, loop=None):
        if max_fps <= 0:
            raise ValueError('max_fps 必須大於 0')
        self.camera = camera
        self.raw_widget = raw_widget
        self.result_widget = result_widget
        self.saved_widget = saved_widget
        self.count_widget = count_widget
        self.status_widget = status_widget
        self.save_status_widget = save_status_widget
        self.dataset_dir = dataset_dir
        self.profile = copy.deepcopy(profile)
        self.max_fps = float(max_fps)
        self.show_midpoint = bool(show_midpoint)
        self.processor = processor or process_camera_frame
        self.loop = loop or asyncio.get_event_loop()
        self.executor = ThreadPoolExecutor(max_workers=1)
        self.task = None
        self.running = False
        self.closed = False
        self.latest = None
        self.frame_id = 0
        self.epoch = 0
        self.bindings = []
        os.makedirs(dataset_dir, exist_ok=True)

    def start(self, button=None):
        if self.closed:
            return
        if not self.running:
            self.epoch += 1
        self.running = True
        self.status_widget.value = '辨識中：等待最新影格…'
        if self.task is None or self.task.done():
            self.task = self.loop.create_task(self._run())

    def pause(self, button=None):
        self.running = False
        self.epoch += 1
        self.status_widget.value = '已暫停，保留最後一幀；可存圖或按開始繼續。'

    async def _run(self):
        try:
            while self.running and not self.closed:
                started = self.loop.time()
                epoch = self.epoch
                frame = self.camera.value
                if (not isinstance(frame, np.ndarray) or frame.ndim != 3
                        or frame.shape[2] != 3 or frame.size == 0):
                    raise ValueError('相機尚未提供有效的 BGR 影格')
                frame = frame.copy()
                captured_at = datetime.datetime.now().isoformat()
                captured_monotonic = time.monotonic()
                raw_bytes, result_bytes, metadata = await self.loop.run_in_executor(
                    self.executor, self.processor, frame, self.profile, self.show_midpoint)
                # 暫停、重建介面或快速重新開始後，不發布舊的運算結果。
                if self.closed or not self.running:
                    break
                if epoch != self.epoch:
                    continue
                self.frame_id += 1
                self.latest = {'raw_jpeg': raw_bytes, 'result_jpeg': result_bytes,
                               'metadata': metadata, 'frame_id': self.frame_id,
                               'captured_at': captured_at, 'captured_monotonic': captured_monotonic, 'size': [frame.shape[1], frame.shape[0]]}
                self.raw_widget.value = raw_bytes
                self.result_widget.value = result_bytes
                elapsed = max(self.loop.time() - started, 0.001)
                midpoint = metadata.get('midpoint')
                self.status_widget.value = (
                    '{} | 第 {} 幀 | {} 組標線 | 單幀 {:.0f} ms | 中點 {}'
                    .format(self.profile.get('name', 'profile_3'), self.frame_id, metadata['lane_group_count'], elapsed * 1000,
                            midpoint if midpoint is not None else '無足夠左右標線'))
                await asyncio.sleep(max(0.0, 1.0 / self.max_fps - elapsed))
        except asyncio.CancelledError:
            pass
        except Exception as exc:
            self.running = False
            if not self.closed:
                self.status_widget.value = '辨識暫停：{}: {}'.format(type(exc).__name__, exc)

    def save(self, click=None):
        """儲存最近發布的同一幀結果；暂停後再點選可固定標註影格。"""
        record = self.latest
        if record is None:
            self.save_status_widget.value = '尚無辨識結果，請先按開始辨識。'
            return
        try:
            click_xy = None
            if click is not None:
                width, height = record['size']
                x = float(click['offsetX'])
                y = float(click['offsetY'])
                if not np.isfinite(x) or not np.isfinite(y):
                    raise ValueError('點選座標無效')
                x = int(round(x * width / float(self.raw_widget.width)))
                y = int(round(y * height / float(self.raw_widget.height)))
                click_xy = [max(0, min(width - 1, x)), max(0, min(height - 1, y))]
            stem = ('xy_%03d_%03d_' % tuple(click_xy) if click_xy is not None else 'lane_') + str(uuid1())
            path = os.path.join(self.dataset_dir, stem + '.jpg')
            with open(path, 'wb') as f:
                f.write(record['result_jpeg'])
            sidecar = {'frame_id': record['frame_id'], 'captured_at': record['captured_at'],
                       'click_xy': click_xy, 'input_size': record['size'], 'profile_id': self.profile.get('name', 'profile_3'),
                       'centerline': record['metadata'].get('centerline'),
                       'midpoint': record['metadata'].get('midpoint'),
                       'lane_group_count': record['metadata']['lane_group_count'],
                       'profile': self.profile}
            with open(os.path.join(self.dataset_dir, stem + '.json'), 'w', encoding='utf-8') as f:
                json.dump(sidecar, f, ensure_ascii=False, indent=2)
            self.saved_widget.value = record['result_jpeg']
            self.count_widget.value = len(glob.glob(os.path.join(self.dataset_dir, '*.jpg')))
            self.save_status_widget.value = '已儲存第 {} 幀：{}'.format(record['frame_id'], stem + '.jpg')
        except Exception as exc:
            self.save_status_widget.value = '存圖失敗：{}: {}'.format(type(exc).__name__, exc)

    def save_button(self, button=None):
        self.save()

    def save_click(self, widget, content, buffers):
        if content.get('event') == 'click':
            self.save(content.get('eventData', {}))

    def shutdown_camera(self, button=None):
        self.close()
        try:
            self.camera.stop()
            self.status_widget.value = '相機已停止。再次使用請重新啟動 Kernel 並依序執行。'
        except Exception as exc:
            self.status_widget.value = '辨識已停止，但相機停止失敗：{}'.format(exc)

    def bind(self, start_button, pause_button, save_button):
        for button, callback in ((start_button, self.start), (pause_button, self.pause),
                                 (save_button, self.save_button)):
            button.on_click(callback)
            self.bindings.append((button, 'on_click', callback))
        self.raw_widget.on_msg(self.save_click)
        self.bindings.append((self.raw_widget, 'on_msg', self.save_click))

    def close(self):
        if self.closed:
            return
        self.closed = True
        self.running = False
        self.epoch += 1
        for widget, method, callback in self.bindings:
            getattr(widget, method)(callback, remove=True)
        self.bindings = []
        if self.task is not None and not self.task.done():
            self.task.cancel()
        self.executor.shutdown(wait=False)




class DatasetController(LiveLaneController):
    """保存未畫線原圖；標籤與影像必須來自同一個已發布影格。"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.session_id = str(uuid1())
        self.saved_frames = set()
        self.last_save_time = -float('inf')
        self.drive = None
        self.gamepad_binding = None
        self.aux_bindings = []
        self.review_dir = os.path.join(os.path.dirname(self.dataset_dir), 'review')
        self.meta_dir = os.path.join(os.path.dirname(self.dataset_dir), 'metadata')
        for directory in (self.review_dir, self.meta_dir):
            os.makedirs(directory, exist_ok=True)
        self.update_count()

    def update_count(self):
        count = len(glob.glob(os.path.join(self.dataset_dir, 'xy_*.jpg')))
        self.count_widget.value = count
        progress_widget.max = max(TARGET_COUNT, count)
        progress_widget.value = count
        progress_text.value = '{} / {} 張{}'.format(count, TARGET_COUNT,
            '（已達目標，仍可繼續）' if count >= TARGET_COUNT else '')

    def save(self, click=None):
        if self.closed:
            return
        record = self.latest
        if record is None:
            self.save_status_widget.value = '尚無影格，請先開始辨識。'
            return
        now = time.monotonic()
        if self.running and now - record['captured_monotonic'] > MAX_CAPTURE_AGE:
            self.save_status_widget.value = '影格過舊，請等辨識畫面更新後再拍。'
            return
        if record['frame_id'] in self.saved_frames:
            self.save_status_widget.value = '這一幀已存過；請開始辨識取得新畫面。'
            return
        if now - self.last_save_time < CAPTURE_DEBOUNCE:
            return
        try:
            width, height = record['size']
            if click is not None:
                if self.running:
                    raise ValueError('手動標記請先暫停辨識，再點左側原圖的道路目標位置')
                x = float(click['offsetX']) * width / float(self.raw_widget.width)
                y = float(click['offsetY']) * height / float(self.raw_widget.height)
                if not np.isfinite([x, y]).all():
                    raise ValueError('座標無效')
                target = [int(np.clip(round(x), 0, width - 1)),
                          int(np.clip(round(y), 0, height - 1))]
                source = 'manual_click'
            else:
                centerline = record['metadata'].get('centerline', {})
                target = centerline.get('far_point')
                if not centerline.get('valid') or target is None:
                    raise ValueError('沒有有效中點，未儲存；可暫停後點左圖手動標記')
                target = [int(v) for v in target]
                source = 'hough_midpoint_confirmed_by_button'
            if not (0 <= target[0] < width and 0 <= target[1] < height):
                raise ValueError('目標超出圖片範圍')
            stem = 'xy_%03d_%03d_%s' % (target[0], target[1], uuid1())
            image_path = os.path.join(self.dataset_dir, stem + '.jpg')
            review_path = os.path.join(self.review_dir, stem + '.jpg')
            meta_path = os.path.join(self.meta_dir, stem + '.json')
            # 檢查圖不放在訓練資料夾；人工標記另以黃色圈顯示實際標籤。
            review = cv2.imdecode(np.frombuffer(record['result_jpeg'], np.uint8), 1)
            cv2.circle(review, tuple(target), 6, (0, 255, 255), 2)
            review_bytes = bytes(bgr8_to_jpeg(review))
            info = {'session_id': self.session_id, 'frame_id': record['frame_id'],
                    'captured_at': record['captured_at'], 'saved_at': datetime.datetime.now().isoformat(),
                    'target_xy': target, 'label_source': source, 'input_size': record['size'],
                    'centerline': record['metadata'].get('centerline'),
                    'profile': self.profile, 'centerline_config': dict(CENTERLINE_CONFIG),
                    'image_sha256': hashlib.sha256(record['raw_jpeg']).hexdigest()}
            created = []
            try:
                for path, content in ((meta_path, json.dumps(info, ensure_ascii=False, indent=2).encode('utf-8')),
                                      (review_path, review_bytes), (image_path, record['raw_jpeg'])):
                    with open(path, 'xb') as f:
                        created.append(path)
                        f.write(content)
            except Exception:
                for path in created:
                    os.remove(path)
                raise
            self.saved_frames.add(record['frame_id'])
            self.last_save_time = now
            self.saved_widget.value = review_bytes
            self.update_count()
            self.save_status_widget.value = '已存原圖：{}；標籤 {}（{}）'.format(stem + '.jpg', target, source)
        except Exception as exc:
            self.save_status_widget.value = '未儲存：{}'.format(exc)

    def pause(self, button=None):
        if self.drive is not None:
            self.drive.stop()
        super().pause(button)

    async def _run(self):
        try:
            await super()._run()
        finally:
            if self.drive is not None:
                self.drive.stop()

    def close(self):
        if self.closed:
            return
        if self.gamepad_binding is not None:
            self.gamepad_binding.close()
        if self.drive is not None:
            self.drive.stop()
        for widget, method, callback in self.aux_bindings:
            getattr(widget, method)(callback, remove=True)
        self.aux_bindings = []
        super().close()


class PulseDrive:
    """保留原本前進 1 秒的動作，改成按鍵觸發，不自動循線。"""
    def __init__(self, lane, enabled, robot_factory):
        self.lane = lane
        self.enabled = enabled
        self.robot_factory = robot_factory
        self.robot = None
        self.task = None

    def start(self, button=None):
        if self.lane.closed or not self.enabled.value or not self.lane.running:
            motion_status.value = '未移動：請先開始辨識，再勾選允許移動。'
            return
        record = self.lane.latest
        if record is None or time.monotonic() - record['captured_monotonic'] > MAX_CAPTURE_AGE:
            motion_status.value = '未移動：等待新影格。'
            return
        if self.task is not None and not self.task.done():
            return
        self.task = self.lane.loop.create_task(self._pulse())

    async def _pulse(self):
        try:
            if self.robot is None:
                self.robot = self.robot_factory()
            self.robot.forward(MOVE_SPEED)
            motion_status.value = '前進中，可按拍照；到時自動停止。'
            await asyncio.sleep(MOVE_SECONDS)
        except asyncio.CancelledError:
            pass
        except Exception as exc:
            motion_status.value = '移動錯誤：{}'.format(exc)
        finally:
            self.stop_motor()

    def stop_motor(self):
        if self.robot is not None:
            try:
                self.robot.stop()
                motion_status.value = '車子已停止。'
            except Exception as exc:
                motion_status.value = '停止指令失敗：{}'.format(exc)

    def stop(self, button=None):
        if self.task is not None and not self.task.done():
            self.task.cancel()
        self.stop_motor()


class GamepadBinding:
    def __init__(self, lane, controller):
        self.lane = lane
        self.controller = controller
        self.bindings = []
        controller.observe(self.connection_changed, names='connected')
        controller.observe(self.buttons_changed, names='buttons')
        move_enabled.observe(self.enable_changed, names='value')

    def enable_changed(self, change):
        if not change['new']:
            self.lane.drive.stop()

    def connection_changed(self, change):
        if not change['new']:
            self.unbind()
            move_enabled.value = False
            self.lane.drive.stop()
            gamepad_status.value = '手把已斷線，已停止移動；重連後請重新綁定。'

    def buttons_changed(self, change):
        if self.bindings:
            self.unbind()
            move_enabled.value = False
            self.lane.drive.stop()
            gamepad_status.value = '手把按鍵列表已更新，請重新綁定。'

    def unbind(self):
        for button, callback in self.bindings:
            button.unobserve(callback, names='value')
        self.bindings = []

    def bind(self, button=None):
        self.unbind()
        move_enabled.value = False
        self.lane.drive.stop()
        indices = (CAPTURE_BUTTON, MOVE_BUTTON, STOP_BUTTON)
        if len(set(indices)) != 3 or min(indices) < 0:
            gamepad_status.value = '拍照、移動、停止按鍵編號需不同且不小於零。'
            return
        if not self.controller.connected or len(self.controller.buttons) <= max(indices):
            gamepad_status.value = '尚未讀到足夠按鍵：先按手把任一鍵，看到 Controller 回應後再綁定。'
            return
        actions = (self.lane.save_button, self.lane.drive.start, self.stop_all)
        for index, action in zip(indices, actions):
            # 只處理由放開到按下的邊緣，長按不連拍、不重複移動。
            def callback(change, action=action):
                if float(change.get('old', 0)) <= 0.5 < float(change['new']):
                    action()
            target = self.controller.buttons[index]
            target.observe(callback, names='value')
            self.bindings.append((target, callback))
        gamepad_status.value = '已綁定：{} 拍照／{} 前進／{} 停止。'.format(*indices)

    def stop_all(self, button=None):
        move_enabled.value = False
        self.lane.drive.stop()

    def close(self):
        self.unbind()
        self.controller.unobserve(self.connection_changed, names='connected')
        self.controller.unobserve(self.buttons_changed, names='buttons')
        move_enabled.unobserve(self.enable_changed, names='value')


# 重新執行時，解除舊按鍵與背景工作。
previous = globals().get('lane_controller')
if previous is not None:
    previous.close()
lane_controller = DatasetController(
    camera, camera_widget, img_widget, snapshot_widget,
    count_widget, status_widget, save_status_widget, DATASET_DIR,
    LANE_PROFILE, max_fps=MAX_FPS, show_midpoint=True)
lane_controller.drive = PulseDrive(lane_controller, move_enabled, Robot)
lane_controller.gamepad_binding = GamepadBinding(lane_controller, controller)
lane_controller.bind(start_button, pause_button, save_button)
for button, callback in ((stop_camera_button, lane_controller.shutdown_camera),
                         (bind_gamepad_button, lane_controller.gamepad_binding.bind),
                         (move_button, lane_controller.drive.start),
                         (stop_move_button, lane_controller.gamepad_binding.stop_all)):
    button.on_click(callback)
    lane_controller.aux_bindings.append((button, 'on_click', callback))
print('資料集目錄：' + os.path.abspath(DATASET_DIR))
print('預設不移動。先開始辨識、喚醒手把，再按「綁定手把」。')


## 7 顯示與操作
依序執行後，開始辨識 → 手把按任一鍵 → 綁定手把 → 確認綠點 → 13 號鍵拍照。
需要移動時勾選允許移動，再按 5 號鍵；1 號鍵停止並取消允許移動。暫停辨識、停止相機或手把斷線也會發出停止指令。


In [ ]:
data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([start_button, pause_button, save_button, stop_camera_button]),
    ipywidgets.HBox([
        ipywidgets.VBox([ipywidgets.Label(value='目前處理的原圖'), camera_widget]),
        ipywidgets.VBox([ipywidgets.Label(value='動態辨識結果'), img_widget]),
        ipywidgets.VBox([ipywidgets.Label(value='上次標記檢查圖（不供訓練）'), snapshot_widget]),
    ]),
    controller,
    ipywidgets.HBox([bind_gamepad_button, move_enabled, move_button, stop_move_button]),
    gamepad_status,
    motion_status,
    progress_widget,
    progress_text,
    count_widget,
    status_widget,
    save_status_widget,
])
display(data_collection_widget)


## 8 收集完成與訓練銜接
- 訓練照片：`project4_data/dataset_xy/xy_*.jpg`；檢查圖與 JSON 分別位於同層 review、metadata，不送入訓練。
- 訓練 Notebook 的資料路徑改成 `XYDataset('project4_data/dataset_xy', random_hflips=False)`。若傳到另一台電腦，只傳原圖資料夾並指定其路徑；已有資料時跳過範例 unzip。
- 200 張是起點，請包含直路、左右彎、偏左與偏右後需要回復的畫面。不要只在同一位置連拍。每拍完一段檢查 review，錯誤標籤的原圖不要拿去訓練。
- JSON 記錄 session_id、擷取時間、影格編號、標記來源，可用來按拍攝批次切分；避免相鄰近似照片隨機落入兩邊。課堂 90/10 比例在 200 張時約為 180/20。
- 課堂 train_model 的 random_hflips 旗標未實際控制翻轉，而且驗證集共用隨機增強；本次未修改或執行訓練，正式訓練前應另處理。
- 結束按停止相機；此檔不在 Run All 中執行一次性的前進命令。手把連線、按鍵編號及實機移動需在 JetBot 確認，本地測試不能替代硬體測試。
